# Advanced: Multi-Strategy Numeric Generalization with PAMOLA.CORE

**Goal**: Master all 3 numeric generalization strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Binning - Group values into discrete intervals
- **Strategy 2**: Rounding - Reduce precision to significant digits
- **Strategy 3**: Range-based - Map values to custom ranges
- Calculate advanced privacy metrics (k-anonymity, information loss)
- Analyze privacy-utility tradeoffs
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of privacy concepts
- Familiarity with operation.execute() pattern

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── anonymization/generalization/
        └── 02_numeric_generalization_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.anonymization.generalization.numeric_op import NumericGeneralizationOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Load from examples/data_examples/sample.csv
- Or auto-generates 1000-record sample if not found
- Inspect data structure and numeric distributions

**What you'll see:**
- File load status or generation confirmation
- Dataset summary (1000 records, columns count)
- First 5 sample records
- Column statistics (min/max/mean for numeric, unique counts for categorical)

**Dataset features:**
- Multiple numeric fields: age (18-75), salary (30K-200K), score (0-100), experience_years (0-40)
- Categorical fields: department, location

**Note:** Generated data is automatically saved. The notebook uses years_of_experience field for all 3 strategies

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    np.random.seed(42)
    n_records = 1000
    
    df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'age': np.clip(np.random.normal(40, 12, n_records), 18, 75).astype(int),
        'salary': np.clip(np.random.lognormal(11, 0.5, n_records), 30000, 200000).astype(int),
        'score': np.round(np.random.beta(8, 2, n_records) * 100, 2),
        'experience_years': np.clip((np.random.normal(40, 12, n_records) - 22 + np.random.normal(0, 3, n_records)), 0, 40).astype(int),
        'department': np.random.choice(['Engineering', 'Sales', 'HR', 'Marketing', 'Finance', 'Operations'], n_records, p=[0.3, 0.2, 0.1, 0.15, 0.15, 0.1]),
        'location': np.random.choice(['New York', 'San Francisco', 'Chicago', 'Boston', 'Seattle'], n_records, p=[0.3, 0.25, 0.2, 0.15, 0.1])
    })
    
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save (file may be open)")

print(f"\n📊 Dataset: {len(df):,} records, {len(df.columns)} columns")
print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s}: {df[col].nunique()} unique")
    else:
        print(f"  {col:20s}: min={df[col].min():.0f}, max={df[col].max():.0f}, mean={df[col].mean():.1f}")

## Step 3: Setup Shared Environment

**How to use:**
- Create shared task directory for all strategies
- All 3 strategies use the same field: years_of_experience
- Run to initialize shared components

**What you'll see:**
- Field validation results (3 strategies, all using years_of_experience)
- Value range for the field (e.g., 0-40 years)
- Task directory created (✓)
- TaskReporter initialized (✓)
- Config kwargs and DataSource created (✓)

**Note:** Using same field for all strategies allows direct comparison of results and privacy-utility trade-offs. Field must be numeric type.

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_field": "years_of_experience",      # Binning
    "strategy2_field": "years_of_experience",   # Rounding
    "strategy3_field": "years_of_experience"     # Range-based
}

print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns or not pd.api.types.is_numeric_dtype(df[field_name]):
        raise ValueError(f"Field '{field_name}' not found or not numeric!")
    print(f"  ✓ {strategy:20s}: '{field_name}' (range: {df[field_name].min():.0f}-{df[field_name].max():.0f})")

def create_config_kwargs():
    return {"dataset_name": "main_dataset"}

task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

task_reporter = TaskReporter(
    task_id="advanced_numeric_001",
    task_type="multi_strategy_numeric",
    description="Multi-strategy numeric generalization",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

kwargs = create_config_kwargs()
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")
print(f"\n✅ Shared environment ready!")

## STRATEGY 1: Binning (Equal-Width)

**How to use:**
- Groups numeric values into discrete intervals (bins)
- Equal-width: All bins have the same range size
- Run to execute binning strategy

**Key parameters:**
- `strategy='binning'`: Use binning generalization
- `bin_count=5`: Create 5 equal-width bins
- `binning_method='equal_width'`: Uniform bin widths (e.g., 0-8, 8-16, 16-24, 24-32, 32-40)
- `mode='ENRICH'`: Keeps original + adds binned field

**What you'll see:**
- Configuration confirmation
- Progress bar: validation → binning → labeling → metrics → visualize → save
- Completion message with execution time (1-3 seconds)
- Reduction statistics (e.g., 41 → 5 bins, 87.8% reduction)

**Validate:**
- ✅ Execution time <5 seconds
- ✅ Bin count = 5 (as configured)
- ✅ Reduction ≥80%
- ✅ Status = "completed"

**Note:** Best for uniformly distributed data. Use `equal_frequency` for skewed distributions.

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: BINNING (EQUAL-WIDTH)")
print("=" * 80 + "\n")

tracker_s1 = HierarchicalProgressTracker(total=6, description="Strategy 1: Binning", unit="steps")

operation_s1 = NumericGeneralizationOperation(
    field_name=FIELD_CONFIG["strategy1_field"],
    mode='ENRICH',
    output_field_name=f"{FIELD_CONFIG['strategy1_field']}_binned",
    strategy='binning',
    bin_count=5,
    binning_method='equal_width',
    use_vectorization=True,
    generate_visualization=True,
    save_output=True
)

print("✓ Strategy 1 configured\n⚙️  Executing...\n")
start_time = time.time()

result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_binning',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

output_files_s1 = sorted(list((task_dir / 'strategy1_binning' / 'output').glob('*.csv')), key=lambda x: x.stat().st_mtime, reverse=True)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    field_s1 = FIELD_CONFIG["strategy1_field"]
    output_field_s1 = f"{field_s1}_binned"
    reduction = (1 - result_df_s1[output_field_s1].nunique() / result_df_s1[field_s1].nunique()) * 100
    print(f"📊 {result_df_s1[field_s1].nunique()} → {result_df_s1[output_field_s1].nunique()} bins ({reduction:.1f}% reduction)")

## STRATEGY 2: Rounding (Precision Reduction)

**How to use:**
- Reduces precision by rounding to fewer decimal places
- Maintains numeric nature of data (not categorical)
- Run to execute rounding strategy

**Key parameters:**
- `strategy='rounding'`: Use rounding generalization
- `precision=0`: Round to integers (0 decimals)
  - `precision=1`: Round to 1 decimal (45.67 → 45.7)
  - `precision=-1`: Round to tens (45 → 50)
  - `precision=-2`: Round to hundreds (145 → 100)
- `mode='ENRICH'`: Keeps original + adds rounded field

**What you'll see:**
- Configuration confirmation
- Progress bar: validation → rounding → grouping → metrics → visualize → save
- Completion message with execution time (1-3 seconds)
- Reduction statistics (unique values before/after, % reduction)

**Validate:**
- ✅ Execution time <5 seconds
- ✅ Values properly rounded
- ✅ Numeric type preserved
- ✅ Status = "completed"

**Note:** Best for high-precision values. Lower reduction than binning but preserves numeric relationships.

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: ROUNDING")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(total=6, description="Strategy 2: Rounding", unit="steps")

operation_s2 = NumericGeneralizationOperation(
    field_name=FIELD_CONFIG["strategy2_field"],
    mode='ENRICH',
    output_field_name=f"{FIELD_CONFIG['strategy2_field']}_rounded",
    strategy='rounding',
    precision=0,
    use_vectorization=True,
    generate_visualization=True,
    save_output=True
)

print("✓ Strategy 2 configured\n⚙️  Executing...\n")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_rounding',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

output_files_s2 = sorted(list((task_dir / 'strategy2_rounding' / 'output').glob('*.csv')), key=lambda x: x.stat().st_mtime, reverse=True)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    field_s2 = FIELD_CONFIG["strategy2_field"]
    output_field_s2 = f"{field_s2}_rounded"
    reduction = (1 - result_df_s2[output_field_s2].nunique() / result_df_s2[field_s2].nunique()) * 100
    print(f"📊 {result_df_s2[field_s2].nunique()} → {result_df_s2[output_field_s2].nunique()} values ({reduction:.1f}% reduction)")

## STRATEGY 3: Range-Based Generalization

**How to use:**
- Maps values to custom-defined ranges
- Uses domain knowledge to set meaningful thresholds
- Run to execute range-based strategy

**Key parameters:**
- `strategy='range'`: Use range-based generalization
- `range_limits`: Custom ranges defined as list of tuples
  - Example (experience): [(0,5), (5,10), (10,20), (20,40)] → "Junior", "Mid-level", "Senior", "Expert"
  - Current config: [(0,50), (50,75), (75,90), (90,100)]
- `mode='ENRICH'`: Keeps original + adds range field

**What you'll see:**
- Configuration confirmation
- Progress bar: validation → mapping → labeling → metrics → visualize → save
- Completion message with execution time (1-3 seconds)
- Reduction statistics (unique values → ranges, % reduction)

**Validate:**
- ✅ Execution time <5 seconds
- ✅ Values mapped to defined ranges
- ✅ No unmapped values (all within range limits)
- ✅ Status = "completed"

**Note:** Best for domain-defined thresholds. Most flexible - handles skewed distributions and incorporates business logic.

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: RANGE-BASED")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(total=6, description="Strategy 3: Range", unit="steps")

score_ranges = [(0, 50), (50, 75), (75, 90), (90, 100)]

operation_s3 = NumericGeneralizationOperation(
    field_name=FIELD_CONFIG["strategy3_field"],
    mode='ENRICH',
    output_field_name=f"{FIELD_CONFIG['strategy3_field']}_ranged",
    strategy='range',
    range_limits=score_ranges,
    use_vectorization=True,
    generate_visualization=True,
    save_output=True
)

print("✓ Strategy 3 configured\n⚙️  Executing...\n")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_range',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

output_files_s3 = sorted(list((task_dir / 'strategy3_range' / 'output').glob('*.csv')), key=lambda x: x.stat().st_mtime, reverse=True)
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    field_s3 = FIELD_CONFIG["strategy3_field"]
    output_field_s3 = f"{field_s3}_ranged"
    reduction = (1 - result_df_s3[output_field_s3].nunique() / result_df_s3[field_s3].nunique()) * 100
    print(f"📊 {result_df_s3[field_s3].nunique()} → {result_df_s3[output_field_s3].nunique()} ranges ({reduction:.1f}% reduction)")

## Step 4: Strategy Comparison

**How to use:**
- Review execution times for all 3 strategies
- Compare reduction ratios and effectiveness
- Understand when to use each approach

**What you'll see:**
- Execution time for each strategy (seconds)
- Total processing time
- Privacy reduction comparison (unique values before → after)
- Strategy selection guidance

**Strategy selection guide:**
- **Binning**: Uniform distribution + need consistent intervals
- **Rounding**: Preserve numeric type + moderate privacy
- **Range**: Domain expertise + flexible thresholds

**Validate:**
- ✅ All strategies completed successfully
- ✅ Execution times <5 seconds each
- ✅ Progressive reduction visible
- ✅ Total time <15 seconds

**Note:** Fastest strategy isn't always best - consider privacy needs and data utility balance

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

if all(v in locals() for v in ['elapsed_s1', 'elapsed_s2', 'elapsed_s3']):
    print("⏱️  Execution Time:")
    print(f"  Binning:   {elapsed_s1:6.2f}s")
    print(f"  Rounding:  {elapsed_s2:6.2f}s")
    print(f"  Range:     {elapsed_s3:6.2f}s")
    print(f"  Total:     {elapsed_s1+elapsed_s2+elapsed_s3:6.2f}s")

print("\n💡 Key Insights:")
print("  • Binning: Best for uniformly distributed data")
print("  • Rounding: Best for high-precision values")
print("  • Range: Best for domain-defined thresholds")

## Step 5: Detailed Artifact Review (All Strategies)

**How to use:**
- Review artifacts from all 3 strategies in one place
- Auto-loads NEWEST files from each strategy
- Displays metrics, mappings, and visualizations inline

**What you'll see (per strategy):**
1. **Metrics JSON**: Effectiveness ratio, performance metrics, numeric information loss (normalized loss, avg error)
2. **Mapping Files**: Original → generalized value transformations, changed vs unchanged count
3. **Visualizations**: Before/after charts from latest batch (first 2 displayed)

**Note:** Only newest files shown. Multiple test runs create versions - older artifacts excluded automatically

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_binning', 'Strategy 1: Binning'),
    ('strategy2_rounding', 'Strategy 2: Rounding'),
    ('strategy3_range', 'Strategy 3: Range-Based')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        if metrics_files:
            # 1) Exclude data_types inside IF
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                # Use only filtered files
                target_files = filtered
                print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
            else:
                # Fallback: nothing left after filtering → use all
                target_files = metrics_files
                print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

            # 2) Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]

            print(f"📄 Loading LATEST: {latest_metrics_file.name}")
            mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
            print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
            print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
            print("=" * 80)

            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                    
                if 'effectiveness' in metrics:
                    eff = metrics['effectiveness']
                    print(f"   Original: {eff.get('original_unique')} → Anonymized: {eff.get('anonymized_unique')}")
                    print(f"   Reduction: {eff.get('effectiveness_ratio', 0):.2%}")
                    
                if 'performance' in metrics:
                    perf = metrics['performance']
                    print(f"   Duration: {perf.get('duration_seconds', 0):.4f}s")
                    
                if 'numeric_info_loss' in metrics:
                    info = metrics['numeric_info_loss']
                    print(f"   Info Loss: {info.get('normalized_loss', 0):.2%}")
                    print(f"   Avg Error: {info.get('avg_absolute_error', 0):.2f}")
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Mapping (NEWEST)
    mapping_files = sorted(list(strategy_dir.glob('**/*mapping*.json')),
                          key=lambda x: x.stat().st_mtime, reverse=True)
    if mapping_files:
        print(f"\n📄 MAPPING: {mapping_files[0].name}")
        try:
            with open(mapping_files[0], 'r') as f:
                mapping_data = json.load(f)
                mappings = mapping_data.get('mappings', {})
                changed = sum(1 for k, v in mappings.items() if float(k) != v)
                print(f"   Total: {len(mappings)}, Changed: {changed}, Unchanged: {len(mappings)-changed}")
        except Exception as e:
            print(f"   ⚠️  Error: {e}")
    
    # 3. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(list(viz_dir.glob('*.png')),
                          key=lambda x: x.stat().st_mtime, reverse=True)
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [f for f in viz_files if abs(f.stat().st_mtime - latest_time) < 10]
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Calculate Privacy Metrics

**How to use:**
- Calculate privacy metrics for each strategy
- Compare privacy protection levels
- Run AFTER all 3 strategies complete

**What you'll see (per strategy):**
- Original vs anonymized unique values
- Min/avg/max group size (k-anonymity metrics)
- Vulnerable groups count (k < 5)
- Strategy comparison table with recommendations

**Privacy targets:**
- Min group size ≥ 5: Basic privacy
- Min group size ≥ 10: Strong privacy
- Zero vulnerable groups: Ideal for production

**Note:** Lower unique values = better privacy but less utility. Balance based on your requirements

In [ ]:
print("\n" + "=" * 80)
print("🔒 PRIVACY METRICS")
print("=" * 80 + "\n")

def calculate_numeric_privacy(df, field_name):
    """Calculate privacy metrics for numeric fields"""
    unique_count = df[field_name].nunique()
    group_sizes = df.groupby(field_name).size()
    
    return {
        'unique_values': int(unique_count),
        'min_group_size': int(group_sizes.min()),
        'avg_group_size': float(group_sizes.mean()),
        'max_group_size': int(group_sizes.max()),
        'vulnerable_groups': int((group_sizes < 5).sum())
    }

# Check if strategies completed
try:
    field_s1 = FIELD_CONFIG["strategy1_field"]
    field_s2 = FIELD_CONFIG["strategy2_field"]
    field_s3 = FIELD_CONFIG["strategy3_field"]
    
    # Strategy 1: Binning
    if 'result_df_s1' in locals() and result_df_s1 is not None:
        print("📊 STRATEGY 1: BINNING")
        output_field_s1 = f"{field_s1}_binned"
        
        privacy_orig_s1 = calculate_numeric_privacy(df, field_s1)
        privacy_anon_s1 = calculate_numeric_privacy(result_df_s1, output_field_s1)
        
        print(f"   Original unique values: {privacy_orig_s1['unique_values']}")
        print(f"   Anonymized unique values: {privacy_anon_s1['unique_values']}")
        print(f"   Min group size: {privacy_anon_s1['min_group_size']}")
        print(f"   Avg group size: {privacy_anon_s1['avg_group_size']:.2f}")
        print(f"   Vulnerable groups (k<5): {privacy_anon_s1['vulnerable_groups']}")
    
    # Strategy 2: Rounding
    if 'result_df_s2' in locals() and result_df_s2 is not None:
        print(f"\n📊 STRATEGY 2: ROUNDING")
        output_field_s2 = f"{field_s2}_rounded"
        
        privacy_orig_s2 = calculate_numeric_privacy(df, field_s2)
        privacy_anon_s2 = calculate_numeric_privacy(result_df_s2, output_field_s2)
        
        print(f"   Original unique values: {privacy_orig_s2['unique_values']}")
        print(f"   Anonymized unique values: {privacy_anon_s2['unique_values']}")
        print(f"   Min group size: {privacy_anon_s2['min_group_size']}")
        print(f"   Avg group size: {privacy_anon_s2['avg_group_size']:.2f}")
        print(f"   Vulnerable groups (k<5): {privacy_anon_s2['vulnerable_groups']}")
    
    # Strategy 3: Range-based
    if 'result_df_s3' in locals() and result_df_s3 is not None:
        print(f"\n📊 STRATEGY 3: RANGE-BASED")
        output_field_s3 = f"{field_s3}_ranged"
        
        privacy_orig_s3 = calculate_numeric_privacy(df, field_s3)
        privacy_anon_s3 = calculate_numeric_privacy(result_df_s3, output_field_s3)
        
        print(f"   Original unique values: {privacy_orig_s3['unique_values']}")
        print(f"   Anonymized unique values: {privacy_anon_s3['unique_values']}")
        print(f"   Min group size: {privacy_anon_s3['min_group_size']}")
        print(f"   Avg group size: {privacy_anon_s3['avg_group_size']:.2f}")
        print(f"   Vulnerable groups (k<5): {privacy_anon_s3['vulnerable_groups']}")
    
    # Summary comparison
    if all(v in locals() for v in ['privacy_anon_s1', 'privacy_anon_s2', 'privacy_anon_s3']):
        print(f"\n" + "=" * 80)
        print("🎯 STRATEGY COMPARISON")
        print("=" * 80)
        print(f"\n{'Strategy':<20s} {'Unique Values':<15s} {'Avg Group':<12s} {'Info Loss':<12s}")
        
        print(f"\n💡 Recommendations:")
        best_privacy = min(privacy_anon_s1['unique_values'], privacy_anon_s2['unique_values'], privacy_anon_s3['unique_values'])
        if best_privacy == privacy_anon_s1['unique_values']:
            print("   • Binning provides strongest privacy protection")
        elif best_privacy == privacy_anon_s2['unique_values']:
            print("   • Rounding provides strongest privacy protection")
        else:
            print("   • Range-based provides strongest privacy protection")
        
        print("   • Lower unique values = better privacy")
        print("   • Higher information loss = reduced data utility")
        print("   • Balance based on your requirements")
        
except NameError:
    print("⚠️  FIELD_CONFIG not defined!")
    print("   Please run 'Step 4: Setup Shared Environment' cell first")
    print("   Then run all 3 strategy cells before calculating privacy metrics")

## Step 7: Export Final Data

**How to use:**
- Export final anonymized datasets from ALL 3 strategies
- Each strategy gets its own output file
- Run AFTER all 3 strategies complete

**What you'll see:**
- Export confirmation for each strategy (✅ or ⚠️ permission warning)
- Final directory location

**Output structure:**
```
advanced_output/
├── strategy1_binning/binning_data.csv
├── strategy2_rounding/rounding_data.csv
└── strategy3_range/range_data.csv
```

**Note:** Files include original and generalized numeric fields. If file is open elsewhere, you'll see a permission warning

In [ ]:
print("=" * 80)
print("📦 EXPORTING FINAL DATA")
print("=" * 80 + "\n")

for i, (result_df, strategy_name, field_name) in enumerate([
    (result_df_s1 if 'result_df_s1' in locals() else None, 'binning', FIELD_CONFIG["strategy1_field"]),
    (result_df_s2 if 'result_df_s2' in locals() else None, 'rounding', FIELD_CONFIG["strategy2_field"]),
    (result_df_s3 if 'result_df_s3' in locals() else None, 'range', FIELD_CONFIG["strategy3_field"])
], 1):
    if result_df is None:
        continue
        
    s_dir = task_dir / f'strategy{i}_{strategy_name}'
    output_path = s_dir / f'{strategy_name}_data.csv'
    
    try:
        export_cols = [c for c in ['record_id', field_name, f'{field_name}_{strategy_name[:6]}'] if c in result_df.columns]
        result_df[export_cols].to_csv(output_path, index=False)
        print(f"✅ Strategy {i}: {output_path.name}")
    except PermissionError:
        print(f"⚠️  Strategy {i}: Cannot save (file may be open)")

print(f"\n📂 All files in: {task_dir}")

## 🎯 Summary

**Accomplished:**
- ✅ Binning: Equal-width intervals
- ✅ Rounding: Precision reduction
- ✅ Range-based: Custom thresholds
- ✅ Performance comparison
- ✅ Data export

**Next steps:**
- Try different bin counts/precision levels
- Experiment with equal_frequency binning
- Define custom ranges for your domain
- Combine with categorical operations

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)